In [0]:
%run "../includes/configuration"

In [0]:
movies_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movies")
).select("movie_id", "budget", "revenue", "year_release_date")


In [0]:
movies_genres_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movie_genre")
).select("genre_id", "movie_id")

In [0]:
genres_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/genres")
).select("genre_id", "genre_name")

In [0]:

results_group_movie_genre = (
    movies_df.filter(movies_df["year_release_date"] > 2015)
    .join(movies_genres_df,
        movies_df["movie_id"] == movies_genres_df["movie_id"], "inner")
    .join(genres_df,
            movies_genres_df["genre_id"] == genres_df["genre_id"])
).select("budget", "revenue", "year_release_date", "genre_name")

In [0]:
from pyspark.sql.functions import sum, avg, max, min, rank, dense_rank,  desc, col
from pyspark.sql.window import Window

In [0]:
budget_rank = Window.partitionBy("year_release_date").orderBy(desc("total_budget"))
revenue_rank = Window.partitionBy("year_release_date").orderBy(desc("total_revenue"))
budget_revenue_rank = (
    Window.partitionBy("year_release_date")
        .orderBy(
            desc("total_budget"), desc("total_revenue"))
)

results_group_movie_genre = (
    results_group_movie_genre
    .groupBy("year_release_date","genre_name")
        .agg(
            sum("budget").alias("total_budget"),
            sum("revenue").alias("total_revenue")  
        )
    .withColumn("budget_rank", rank().over(budget_rank))
    .withColumn("revenue_dense_rank", dense_rank().over(revenue_rank))
    .withColumn("budget_revenue_rank", rank().over(budget_revenue_rank))
)
display(results_group_movie_genre)

In [0]:
(
    results_group_movie_genre
    .write
        .mode("overwrite")
        .format("parquet")
        .save(f"{gold_folder_path}/results_group_movie_genre")
)